# Stage 9H — Diagnose HEAT/HOT WATER before changing the model

**Goal:** Find out what the earlier data can tell us about the missed HEAT/HOT WATER cases.

**Why:** We should investigate data coverage, monthly outcomes, and model probabilities before choosing a fix. We use the train-only candidate models on January–February validation, so the diagnostic predictions are out of sample. We do not reopen March–April test rows or use their labels to select a new threshold.

**Important:** The current release decision remains HOLD_FOR_OPERATIONAL_USE.

In [ ]:
# Cell 2 — Why: verify earlier data and saved candidate models.
from google.colab import drive
from pathlib import Path
import hashlib, json, pandas as pd, numpy as np, joblib, sklearn

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
split_id = "20260926T142339294958Z"
experiment_id = "20260926T143147282446Z"
split_dir = root / "splits" / split_id
experiment_dir = root / "experiments" / experiment_id
split = json.loads((split_dir / "manifest.json").read_text())
experiment = json.loads((experiment_dir / "experiment.json").read_text())

if split["split_id"] != split_id or experiment["split_id"] != split_id:
    raise ValueError("Wrong split or experiment")
if sklearn.__version__ != experiment["versions"]["scikit_learn"]:
    raise ValueError("scikit-learn version changed")

frames = {}
for name in ["train", "validation"]:
    file = split_dir / f"{name}.jsonl"
    raw = file.read_bytes()
    if hashlib.sha256(raw).hexdigest() != split["files"][name]["sha256"]:
        raise ValueError(f"{name} file changed")
    frames[name] = pd.read_json(file, lines=True)
    if len(frames[name]) != split["files"][name]["rows"]:
        raise ValueError(f"{name} row count changed")
train, valid = frames["train"], frames["validation"]

models = {}
for name in ["logistic", "tree"]:
    file = experiment_dir / experiment["artifacts"][name]["file"]
    if hashlib.sha256(file.read_bytes()).hexdigest() != experiment["artifacts"][name]["sha256"]:
        raise ValueError(f"{name} candidate changed")
    models[name] = joblib.load(file)

print("Train rows:", len(train), "Validation rows:", len(valid))
print("Candidate models verified:", list(models))
print("March–April test remains retired")

In [ ]:
# Cell 3 — Why: see when HEAT/HOT WATER occurs and how often it is late.
def heat_months(frame):
    heat = frame[frame["complaint_type"].eq("HEAT/HOT WATER")]
    table = heat.groupby("month")["label"].agg(rows="size", late_cases="sum")
    table["late_rate"] = (table["late_cases"] / table["rows"]).round(3)
    return table.reset_index()

train_heat_months = heat_months(train)
valid_heat_months = heat_months(valid)
print("2025 training HEAT/HOT WATER by month:")
display(train_heat_months)
print("January–February 2026 validation HEAT/HOT WATER by month:")
display(valid_heat_months)

In [ ]:
# Cell 4 — Why: inspect validation probabilities before choosing a fix.
features = split["features"]
heat_mask = valid["complaint_type"].eq("HEAT/HOT WATER").to_numpy()
heat_labels = valid.loc[heat_mask, "label"].to_numpy()
probabilities = {
    name: model.predict_proba(valid[features])[:, 1][heat_mask]
    for name, model in models.items()
}

for name, probs in probabilities.items():
    print("\n", name, "HEAT/HOT WATER validation rows:", len(probs))
    print("Probability percentiles 10/25/50/75/90:",
          np.quantile(probs, [0.10, 0.25, 0.50, 0.75, 0.90]).round(3))
    for threshold in [0.10, 0.20, 0.30]:
        pred = probs >= threshold
        true_late = int((pred & (heat_labels == 1)).sum())
        print("threshold", threshold,
              "flagged", int(pred.sum()),
              "late found", true_late,
              "recall", round(true_late / int(heat_labels.sum()), 3),
              "precision", round(true_late / int(pred.sum()), 3)
              if pred.sum() else None)

In [ ]:
# Cell 5 — Why: locate tree-model misses within validation months.
heat_valid = valid.loc[heat_mask, ["month", "label"]].copy()
heat_valid["probability"] = probabilities["tree"]
heat_valid["flagged"] = heat_valid["probability"] >= 0.20
heat_valid["missed_late"] = heat_valid["label"].eq(1) & ~heat_valid["flagged"]

monthly_errors = heat_valid.groupby("month").agg(
    rows=("label", "size"),
    late_cases=("label", "sum"),
    flagged=("flagged", "sum"),
    missed_late=("missed_late", "sum"),
    mean_probability=("probability", "mean"),
).reset_index()
monthly_errors["mean_probability"] = monthly_errors["mean_probability"].round(3)
print("Tree at the frozen 0.20 threshold, validation only:")
display(monthly_errors)

In [ ]:
# Cell 6 — Why: save a reproducible diagnosis, not a revised model.
from datetime import datetime, timezone

diagnosis_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
diagnosis_dir = root / "diagnoses" / diagnosis_id
diagnosis_dir.mkdir(parents=True, exist_ok=False)

def records(table):
    return json.loads(table.to_json(orient="records"))

summary = {}
for name, probs in probabilities.items():
    summary[name] = {
        "percentiles_10_25_50_75_90": [float(round(x, 3)) for x in
                                       np.quantile(probs, [0.10, 0.25, 0.50, 0.75, 0.90])],
        "thresholds": []
    }
    for threshold in [0.10, 0.20, 0.30]:
        pred = probs >= threshold
        found = int((pred & (heat_labels == 1)).sum())
        flagged = int(pred.sum())
        summary[name]["thresholds"].append({
            "threshold": threshold, "flagged": flagged, "late_found": found,
            "recall": round(found / int(heat_labels.sum()), 3),
            "precision": round(found / flagged, 3) if flagged else None
        })

diagnosis = {
    "diagnosis_id": diagnosis_id,
    "split_id": split_id,
    "experiment_id": experiment_id,
    "train_sha256": split["files"]["train"]["sha256"],
    "validation_sha256": split["files"]["validation"]["sha256"],
    "train_heat_by_month": records(train_heat_months),
    "validation_heat_by_month": records(valid_heat_months),
    "validation_heat_model_summary": summary,
    "validation_heat_tree_monthly_errors": records(monthly_errors),
    "test_rows_used": False,
    "operational_status": "HOLD_FOR_OPERATIONAL_USE",
    "lesson_version": "stage9h-v1"
}
file = diagnosis_dir / "diagnosis.json"
file.write_text(json.dumps(diagnosis, indent=2) + "\n", encoding="utf-8")
print("Diagnosis ID:", diagnosis_id)
print("Saved:", file)
print("No model or threshold was changed")

# Stage 9H takeaway and quiz

**What you learned:** Diagnosis comes before another model. Monthly label rates show whether the problem varies by season. Probabilities show whether the model is uncertain or confidently wrong. The validation-only threshold table helps us see the tradeoff, but March–April is not available for tuning a replacement.

**Easy quiz:**

1. Why score validation with the train-only candidate models instead of the final model refit on train plus validation?
2. If many late HEAT/HOT WATER cases have probabilities below 0.20, what does that explain?
3. Why is lowering the threshold likely to create more false alarms?
4. Why will a revised model need a new, later untouched test period?

Send me both monthly tables, the probability and threshold results from Cell 4, the monthly error table, and the Diagnosis ID. Then we can choose a specific improvement based on evidence.